start off adding libraries for data processing

In [230]:
import pandas as pd

Load and combine the Amazon, IMDb, and Yelp labelled sentence files.
Display the dataset size and class distribution.

In [231]:
amazon = pd.read_csv('C:\\Users\\jna\\Desktop\\Sentiment labelled sentences\\sentiment+labelled+sentences\\sentiment labelled sentences\\amazon_cells_labelled.txt', sep='\t', header=None, names=['sentence', 'label'])
imdb = pd.read_csv('C:\\Users\\jna\\Desktop\\Sentiment labelled sentences\\sentiment+labelled+sentences\\sentiment labelled sentences\\imdb_labelled.txt', sep='\t', header=None, names=['sentence', 'label'])
yelp = pd.read_csv('C:\\Users\\jna\\Desktop\\Sentiment labelled sentences\\sentiment+labelled+sentences\\sentiment labelled sentences\\yelp_labelled.txt', sep='\t', header=None, names=['sentence', 'label'])

combined = pd.concat([amazon, imdb, yelp], ignore_index=True)

print('info',combined.info())
print('head',combined.head())
print('shape',combined.shape)
print(combined['label'].value_counts)

<class 'pandas.DataFrame'>
RangeIndex: 2748 entries, 0 to 2747
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   sentence  2748 non-null   str  
 1   label     2748 non-null   int64
dtypes: int64(1), str(1)
memory usage: 43.1 KB
info None
head                                             sentence  label
0  So there is no way for me to plug it in here i...      0
1                        Good case, Excellent value.      1
2                             Great for the jawbone.      1
3  Tied to charger for conversations lasting more...      0
4                                  The mic is great.      1
shape (2748, 2)
<bound method IndexOpsMixin.value_counts of 0       0
1       1
2       1
3       0
4       1
       ..
2743    0
2744    0
2745    0
2746    0
2747    0
Name: label, Length: 2748, dtype: int64>


Check for missing values and duplicate sentences.

In [232]:
missing=combined.isnull().sum()
print(missing)
duplicates= combined.duplicated().sum()
print(duplicates)
combined = combined.drop_duplicates()
combined = combined.dropna()
print (combined)

sentence    0
label       0
dtype: int64
17
                                               sentence  label
0     So there is no way for me to plug it in here i...      0
1                           Good case, Excellent value.      1
2                                Great for the jawbone.      1
3     Tied to charger for conversations lasting more...      0
4                                     The mic is great.      1
...                                                 ...    ...
2743  I think food should have flavor and texture an...      0
2744                           Appetite instantly gone.      0
2745  Overall I was not impressed and would not go b...      0
2746  The whole experience was underwhelming, and I ...      0
2747  Then, as if I hadn't wasted enough of my life ...      0

[2731 rows x 2 columns]


Convert text to lowercase.

In [233]:
combined['sentence']=combined['sentence'].str.lower()
print(combined.head())

                                            sentence  label
0  so there is no way for me to plug it in here i...      0
1                        good case, excellent value.      1
2                             great for the jawbone.      1
3  tied to charger for conversations lasting more...      0
4                                  the mic is great.      1


Remove punctuation and special characters.

In [234]:
combined['sentence'] = combined['sentence'].str.strip().str.replace(r'[^\w\s\s+]','',regex=True)
print (combined.head())

                                            sentence  label
0  so there is no way for me to plug it in here i...      0
1                          good case excellent value      1
2                              great for the jawbone      1
3  tied to charger for conversations lasting more...      0
4                                   the mic is great      1


Use the sentence text as X and the sentiment label as y.

In [235]:
x = combined['sentence']
y = combined['label']

Split the data into 80% training and 20% testing using stratification.

In [236]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

Convert the sentences into numerical word-count features

In [237]:
from sklearn.feature_extraction.text import CountVectorizer
Vector = CountVectorizer()
x_train = Vector.fit_transform(x_train)
x_test = Vector.transform(x_test)

Train a Multinomial Naive Bayes classifier, Use Laplace smoothing with alpha = 1.

In [238]:
from sklearn.naive_bayes import MultinomialNB
nb_model = MultinomialNB(alpha=1.0)
nb_model.fit(x_train, y_train)
y_pred_nb = nb_model.predict(x_test)

Evaluate the test set using Accuracy, Confusion Matrix, Precision, Recall, and F1-score.

In [239]:
from sklearn.metrics import accuracy_score,confusion_matrix,f1_score,recall_score,precision_score
accuracy_nb = accuracy_score(y_test, y_pred_nb)
precision_nb = precision_score(y_test, y_pred_nb)
f1_nb = f1_score(y_test, y_pred_nb)
recall_nb = recall_score(y_test, y_pred_nb)
cm_nb = confusion_matrix(y_test, y_pred_nb)
print('accuracy_nb : ',accuracy_nb,'\nprecision_nb : ',precision_nb,'\nf1_nb : ',f1_nb,'\nrecall_nb : ',recall_nb,'\ncm_nb : \n',cm_nb)

accuracy_nb :  0.7952468007312614 
precision_nb :  0.8014705882352942 
f1_nb :  0.7956204379562044 
recall_nb :  0.7898550724637681 
cm_nb : 
 [[217  54]
 [ 58 218]]


Classify these two new sentences:
• "The product is excellent and works perfectly."
• "The service was terrible and I will never come back."

In [240]:
new_sentences = [
    "The product is excellent and works perfectly.",
    "The service was terrible and I will never come back."
]
new_sentences = Vector.transform(new_sentences)
predictions = nb_model.predict(new_sentences)
for pred in predictions :
    if pred == 1:
     print('positive')
else:
     print('negative')

positive
negative


## 📊 Model Evaluation & Results Report

### 1. Final Test Accuracy
* **Overall Accuracy:** **79.5%** (0.795)

---

### 2. Performance Comparison (Positive vs. Negative Classes)

| Class | Precision | Recall | F1-Score | Support |
| :--- | :---: | :---: | :---: | :---: |
| **Negative (0)** | 0.79 | 0.80 | 0.79 | 271 |
| **Positive (1)** | 0.80 | 0.79 | 0.80 | 276 |

---

### 3. Summary & Performance Insights
* **Balanced Performance:** The Naive Bayes model shows a very well-balanced performance across both classes with an overall accuracy of **~80%**.
* **Class 0 (Negative):** Achieved a slightly higher **Recall (0.80)**, meaning it successfully identifies negative sentiments very well.
* **Class 1 (Positive):** Achieved a slightly higher **Precision (0.80)**, ensuring reliable positive predictions.
* **Conclusion:** The F1-scores ($\approx 0.79 - 0.80$) confirm that the model generalizes robustly and effectively handles the sentiment analysis task without leaning unfairly toward either class.